In [1]:
import pandas as pd
import numpy as np
import networkx as nx

In [4]:
# Subsitute path with string raw folder
# Here I'm using string v12.5

data_path="/home/ubuntu/Data/"

# load physical string data
string_df = pd.read_csv(data_path+"networks/9606.protein.physical.links.v12.5.txt", sep=" ")
string_df.head(1)

,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000040663,191


In [5]:
string_alias = pd.read_csv(data_path+"networks/9606.protein.aliases.v12.5.txt", sep="\t")
string_alias.head(1)

,#string_protein_id,alias,source
0,9606.ENSP00000000233,2B6H,Ensembl_PDB


In [6]:
#Ensembl_HGNC_ensembl_gene_id
#Ensembl
#Ensembl_gene

string_alias[string_alias['source'] == 'Ensembl_gene'].shape

(19699, 3)

In [8]:
# Filter for data_quality 
string_df[(string_df['combined_score'] >= 700)]

,protein1,protein2,combined_score
7,9606.ENSP00000000233,9606.ENSP00000239690,902
15,9606.ENSP00000000233,9606.ENSP00000256682,840
22,9606.ENSP00000000233,9606.ENSP00000262305,917
23,9606.ENSP00000000233,9606.ENSP00000262455,815
43,9606.ENSP00000000233,9606.ENSP00000306010,999
...,...,...,...
1296473,9606.ENSP00000501277,9606.ENSP00000495360,900
1296476,9606.ENSP00000501317,9606.ENSP00000290524,896
1296477,9606.ENSP00000501317,9606.ENSP00000296785,911
1296479,9606.ENSP00000501317,9606.ENSP00000305071,966


In [ ]:
# filter physical string for high confidence

filtered_string = string_df[string_df['combined_score'] >= 700]
filtered_string.shape

(291630, 3)

In [10]:
filtered_string_ensembl_df = (
    filtered_string
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot1'), 
        left_on="protein1",
        right_on="#string_protein_id_prot1",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot2'), 
        left_on="protein2",
        right_on="#string_protein_id_prot2",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'UniProt_DR_GeneID'].drop_duplicates('#string_protein_id').add_suffix('_ent1'), 
        left_on="protein1",
        right_on="#string_protein_id_ent1",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'UniProt_DR_GeneID'].drop_duplicates('#string_protein_id').add_suffix('_ent2'), 
        left_on="protein2",
        right_on="#string_protein_id_ent2",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_HGNC'].drop_duplicates('#string_protein_id').add_suffix('_sym1'), 
        left_on="protein1",
        right_on="#string_protein_id_sym1",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_HGNC'].drop_duplicates('#string_protein_id').add_suffix('_sym2'), 
        left_on="protein2",
        right_on="#string_protein_id_sym2",
        validate='many_to_one',
        how='left'
    )
    .drop(
        ['#string_protein_id_prot1','#string_protein_id_prot2','source_prot1','source_prot2',
         '#string_protein_id_ent2','#string_protein_id_ent1','#string_protein_id_sym2','#string_protein_id_sym1'],
        axis='columns'  
    )
)

# number of rows does not change 
assert filtered_string_ensembl_df.shape[0] == filtered_string.shape[0] # FAILS as some have no symbol 
filtered_string_ensembl_df.head(2)

,protein1,protein2,combined_score,alias_prot1,alias_prot2,alias_ent1,source_ent1,alias_ent2,source_ent2,alias_sym1,source_sym1,alias_sym2,source_sym2
0,9606.ENSP00000000233,9606.ENSP00000239690,902,ENSG00000004059,ENSG00000120526,381,UniProt_DR_GeneID,84955,UniProt_DR_GeneID,ARF5,Ensembl_HGNC,NUDCD1,Ensembl_HGNC
1,9606.ENSP00000000233,9606.ENSP00000256682,840,ENSG00000004059,ENSG00000134287,381,UniProt_DR_GeneID,377,UniProt_DR_GeneID,ARF5,Ensembl_HGNC,ARF3,Ensembl_HGNC


In [11]:
columns_selection = [
    'protein1',
    'protein2',
    'combined_score',
    'alias_prot1',
    'alias_prot2',
    'alias_ent1',
    'alias_ent2',
    'alias_sym1',
    'alias_sym2'
]

filtered_string_ensembl_df[columns_selection].to_parquet("../data/quality_control/string_physical_filtered.parquet",compression="zstd")